In [2]:
#对每个站点生成折线图（X轴时间，站点反照率；Y轴ERA5land反照率======图中包含R.RMSE,BIAS）
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# 读取 CSV 文件
file_path = r"F:\TUB\greenlandProject\ERA5LAND\snowalbedo\all_stations_snowbilinear.csv"
df = pd.read_csv(file_path, parse_dates=['valid_time'])

# 确保输出目录存在
output_dir = r"F:\TUB\greenlandProject\ERA5LAND\ERA5LANDsaplots"
os.makedirs(output_dir, exist_ok=True)

# 计算 RMSE, R, BIAS 的函数
def calculate_metrics(observed, predicted):
    # 去除 NaN 值
    mask = ~np.isnan(observed) & ~np.isnan(predicted)
    observed = observed[mask]
    predicted = predicted[mask]
    
    # 计算 RMSE, R, BIAS
    rmse = np.sqrt(np.mean((observed - predicted) ** 2))
    r = np.corrcoef(observed, predicted)[0, 1]
    bias = np.mean(observed - predicted)
    
    return rmse, r, bias

# 创建一个空的 DataFrame 来存储结果
results = pd.DataFrame(columns=['station', 'RMSE', 'R', 'BIAS'])

# 遍历每个站点
for station, group in df.groupby('station'):
    plt.figure(figsize=(10, 5))
    plt.plot(group['valid_time'], group['asn'], marker='o', linestyle='-', label='asn')
    plt.plot(group['valid_time'], group['interp_asn'], marker='s', linestyle='--', label='interp_asn')
    
    # 计算并添加 RMSE, R, BIAS
    rmse, r, bias = calculate_metrics(group['asn'].values, group['interp_asn'].values)
    
    # 将结果添加到 DataFrame 使用 pd.concat
    result = pd.DataFrame({'station': [station], 'RMSE': [rmse], 'R': [r], 'BIAS': [bias]})
    results = pd.concat([results, result], ignore_index=True)
    
    # 显示 RMSE, R, BIAS 值
    metrics_text = f"RMSE: {rmse:.4f}\nR: {r:.4f}\nBIAS: {bias:.4f}"
    plt.text(0.05, 0.95, metrics_text, transform=plt.gca().transAxes, fontsize=12,
             verticalalignment='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.5'))
    
    plt.xlabel('Valid Time')
    plt.ylabel('Albedo')
    plt.title(f'Station: {station}')
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid()
    
    # 保存图像
    output_file = os.path.join(output_dir, f"{station}.png")
    plt.savefig(output_file, bbox_inches='tight')
    plt.close()

# 保存结果到 CSV 文件
results_file = r"F:\TUB\greenlandProject\ERA5LAND\ERA5LANDsaplots\metrics_results.csv"
results.to_csv(results_file, index=False)

print("所有站点的折线图已生成并保存，并且系数已保存到 CSV 文件！")


C:\Users\11823\AppData\Local\Temp\ipykernel_18120\449855843.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


所有站点的折线图已生成并保存，并且系数已保存到 CSV 文件！


In [1]:
#ERA5_FAL_interpolated_results结果生成，站点数据和ERA5的albedo
import os
import netCDF4 as nc
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
from datetime import datetime

# 配置参数
nc_file = r"F:\TUB\greenlandProject\ERA5\snowalbedo.nc"  # NetCDF文件路径
csv_file = r"F:\TUB\greenlandProject\NEWdataverse_files\allsites_albedoYM_lonlai.csv"  # 站点数据路径
result_dir = r"F:\TUB\greenlandProject\ERA5\ERA5_saplots"  # 结果保存路径
os.makedirs(result_dir, exist_ok=True)  # 创建结果目录

# 自定义日期解析函数
def parse_custom_date(date_str):
    """
    解析不完整的日期字符串（如"2010/1/"），并将其转换为完整的datetime格式。
    """
    try:
        # 如果日期字符串以"/"结尾，补充"1"作为日
        if date_str.endswith('/'):
            date_str += '1'
        # 解析日期
        return pd.to_datetime(date_str, format='%Y/%m/%d')
    except ValueError:
        # 如果解析失败，返回NaT（Not a Time）
        return pd.NaT

# 读取站点数据
def read_station_data():
    # 读取CSV文件
    df = pd.read_csv(csv_file)
    # 使用自定义函数解析valid_time字段
    df['valid_time'] = df['valid_time'].apply(parse_custom_date)
    # 删除无效日期（NaT）的行
    df = df.dropna(subset=['valid_time'])
    # 提取站点信息
    stations = df[['station', 'longitude', 'latitude']].drop_duplicates().to_dict('records')
    return df, stations

# 插值函数
def grid_interp_to_station(lon, lat, data, station_lon, station_lat, method='nearest'):
    """
    将网格数据插值到站点位置。
    """
    # 将 lon 和 lat 转换为二维网格
    grid_lon, grid_lat = np.meshgrid(lon, lat)
    points = np.column_stack((grid_lon.flatten(), grid_lat.flatten()))
    values = data.flatten()
    station_value = griddata(points, values, (station_lon, station_lat), method=method)
    return station_value

# 主处理函数
def process_data():
    # 读取站点数据
    df, stations = read_station_data()
    station_ids = [s['station'] for s in stations]
    station_lons = [s['longitude'] for s in stations]
    station_lats = [s['latitude'] for s in stations]

    # 读取NetCDF文件
    with nc.Dataset(nc_file) as ds:
        # 获取变量
        fal_data = ds.variables['asn'][:]  # 反照率数据
        time_var = ds.variables['valid_time']  # 时间变量
        lon = ds.variables['longitude'][:]  # 经度
        lat = ds.variables['latitude'][:]  # 纬度

        # 将时间转换为datetime格式
        times = nc.num2date(time_var[:], units=time_var.units, calendar=time_var.calendar)

        # 初始化结果列表
        results = []

        # 遍历站点数据中的每个时间点
        for idx, row in df.iterrows():
            station_id = row['station']
            valid_time = row['valid_time']
            station_lon = row['longitude']
            station_lat = row['latitude']

            # 找到NetCDF文件中对应的时间索引
            time_idx = np.where(times == valid_time)[0]
            if len(time_idx) == 0:
                print(f"未找到时间 {valid_time} 的对应数据")
                continue

            # 获取当前时间步长的数据
            data = fal_data[time_idx[0], :, :]
            # 插值到站点位置
            interpolated_value = grid_interp_to_station(lon, lat, data, station_lon, station_lat)
            if interpolated_value is not None:
                results.append({
                    'valid_time': valid_time,
                    'station': station_id,
                    'interpolated_fal': interpolated_value
                })

    # 将结果保存为CSV文件
    results_df = pd.DataFrame(results)
    output_file = os.path.join(result_dir, 'a.csv')
    results_df.to_csv(output_file, index=False)
    print(f"插值结果已保存为: {output_file}")

# 运行主函数
if __name__ == '__main__':
    process_data()

插值结果已保存为: F:\TUB\greenlandProject\ERA5\ERA5_saplots\a.csv


In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# 文件路径
file_path = r"F:\TUB\greenlandProject\ERA5\ERA5_saplots\interpolated_results.csv"
output_dir = r"F:\TUB\greenlandProject\ERA5\ERA5_saplots"

# 创建输出文件夹
os.makedirs(output_dir, exist_ok=True)

# 读取 CSV 文件
df = pd.read_csv(file_path, delimiter=',', header=0)  # 确保第一行作为列名
df.columns = df.columns.str.strip()  # 去除列名前后的空格

# 转换日期格式
df['valid_time'] = pd.to_datetime(df['valid_time'])

# 遍历所有站点
stations = df['station'].unique()
for station in stations:
    station_df = df[df['station'] == station].sort_values(by='valid_time')
    
    # 获取 asn 和 inter_fal
    asn = station_df['asn'].values
    inter_asn = station_df['inter_asn'].values
    
    # 计算统计指标
    R, _ = pearsonr(asn, inter_asn)
    RMSE = np.sqrt(np.mean((asn - inter_asn) ** 2))
    BIAS = np.mean(asn - inter_asn)
    
    # 绘制折线图
    plt.figure(figsize=(8, 6))
    plt.plot(station_df['valid_time'], asn, label='ASN', marker='o')
    plt.plot(station_df['valid_time'], inter_asn, label='Inter_asn', marker='s')
    plt.xlabel('Date')
    plt.ylabel('Values')
    plt.title(f'Station {station} Albedo & ERA5_asn')
    plt.legend()
    
    # 添加文本信息
    stats_text = f'R = {R:.2f}\nRMSE = {RMSE:.4f}\nBIAS = {BIAS:.4f}'
    plt.text(0.05, 0.85, stats_text, transform=plt.gca().transAxes, fontsize=10,
             bbox=dict(facecolor='white', alpha=0.5))
    
    # 旋转横坐标刻度
    plt.xticks(rotation=45)
    
    # 保存图像
    output_path = os.path.join(output_dir, f'{station}_albedo_ERA5_asn.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

print(f"所有站点的折线图已生成并保存在: {output_dir}")


所有站点的折线图已生成并保存在: F:\TUB\greenlandProject\ERA5\ERA5_saplots


In [ ]:
import os
import pandas as pd
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
from datetime import datetime

# 配置参数
nc_file = r"F:\TUB\greenlandProject\ERA5LAND\forecastAlbedo.nc"  # NetCDF文件路径
csv_file = r"F:\TUB\greenlandProject\NEWdataverse_files\allsites_albedoYM_lonlai.csv" # 站点数据路径
result_dir = r"F:\TUB\greenlandProject\ERA5LAND"  # 结果保存路径
os.makedirs(result_dir, exist_ok=True)  # 创建结果目录

# 自定义日期解析函数
def parse_custom_date(date_str):
    try:
        if date_str.endswith('/'):
            date_str += '1'
        return pd.to_datetime(date_str, format='%Y/%m/%d')
    except ValueError:
        return pd.NaT

# 读取站点数据
def read_station_data():
    df = pd.read_csv(csv_file)
    df['valid_time'] = df['valid_time'].apply(parse_custom_date)
    df = df.dropna(subset=['valid_time'])
    stations = df[['station', 'longitude', 'latitude']].drop_duplicates().to_dict('records')
    return df, stations

# 插值函数
def grid_interp_to_station(lon, lat, data, station_lon, station_lat, method='nearest'):
    grid_lon, grid_lat = np.meshgrid(lon, lat)
    points = np.column_stack((grid_lon.flatten(), grid_lat.flatten()))
    values = data.flatten()
    station_value = griddata(points, values, (station_lon, station_lat), method=method)
    return station_value

# 主处理函数
def process_data():
    df, stations = read_station_data()
    station_ids = [s['station'] for s in stations]
    station_lons = [s['longitude'] for s in stations]
    station_lats = [s['latitude'] for s in stations]

    # 使用xarray打开NetCDF文件，并启用Dask分块
    ds = xr.open_dataset(nc_file, chunks={'valid_time': 10, 'latitude': 100, 'longitude': 100})

    # 获取变量
    fal_data = ds['asn']  # 反照率数据
    valid_time_var = ds['valid_time']  # valid_time变量
    lon = ds['longitude']
    lat = ds['latitude']

    # 将valid_time转换为datetime格式
    valid_times = pd.to_datetime(valid_time_var.values)

    # 初始化结果列表
    results = []

    # 遍历站点数据中的每个时间点
    for idx, row in df.iterrows():
        station_id = row['station']
        valid_time = row['valid_time']
        station_lon = row['longitude']
        station_lat = row['latitude']

        # 找到NetCDF文件中对应的valid_time索引
        time_idx = np.where(valid_times == valid_time)[0]
        if len(time_idx) == 0:
            print(f"未找到时间 {valid_time} 的对应数据")
            continue

        # 获取当前时间步长的数据
        data = fal_data.isel(valid_time=time_idx[0]).values  # 按valid_time索引提取数据

        # 插值到站点位置
        interpolated_value = grid_interp_to_station(lon.values, lat.values, data, station_lon, station_lat)
        if interpolated_value is not None:
            results.append({
                'valid_time': valid_time,
                'station': station_id,
                'interpolated_fal': interpolated_value
            })

    # 将结果保存为CSV文件
    results_df = pd.DataFrame(results)
    output_file = os.path.join(result_dir, 'interpolated_results.csv')
    results_df.to_csv(output_file, index=False)
    print(f"插值结果已保存为: {output_file}")

# 运行主函数
if __name__ == '__main__':
    process_data()


In [7]:
import pandas as pd
import numpy as np

# 读取数据
data = pd.read_csv("F:/TUB/greenlandProject/ERA5LAND/merged_allfeature.csv")

# 选择气象数据列
weather_columns = ['asn', 'sp', 'ssrd', 'str', 'strd', 't2m', 'u10', 'v10']

# 获取所有独特的站点
stations = data['station'].unique()

# 创建一个空的 DataFrame 用于保存结果
results = []

# 遍历每个站点
for station in stations:
    # 选择该站点的数据
    station_data = data[data['station'] == station]

    # 解析时间列（假设有时间列，如果没有则需要根据实际数据做调整）
    station_data.loc[:, 'time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'
    
    # 计算均值
    mean_values = station_data[weather_columns].mean()

    # 检查疑似blunder值（通过计算标准差阈值来进行异常值检测）
    blunder_threshold = 3  # 标准差的倍数
    blunders = {}
    for column in weather_columns:
        column_data = station_data[column]
        mean = column_data.mean()
        std = column_data.std()
        
        # 标记疑似blunder值
        blunder_indices = column_data[(column_data > mean + blunder_threshold * std) | 
                                      (column_data < mean - blunder_threshold * std)].index
        blunders[column] = blunder_indices.tolist()

    # 将结果添加到 DataFrame 中
    station_result = {'station': station}
    station_result.update(mean_values.to_dict())  # 添加均值
    # 添加疑似blunder值索引
    for column in weather_columns:
        station_result[f'{column}_blunders'] = blunders[column]
    
    results.append(station_result)

# 将结果转换为 DataFrame
results_df = pd.DataFrame(results)

# 保存结果为 CSV 文件
output_path = "F:\TUB\greenlandProject\ERA5LAND\meanblunder.csv"
results_df.to_csv(output_path, index=False)

print(f"结果已保存到: {output_path}")


结果已保存到: F:\TUB\greenlandProject\ERA5LAND\meanblunder.csv


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取数据
data = pd.read_csv("F:/TUB/greenlandProject/ERA5LAND/merged_allfeature.csv")

# 选择气象数据列
weather_columns = ['asn', 'sp', 'ssrd', 'str', 'strd', 't2m', 'u10', 'v10']

# 获取所有独特的站点
stations = data['station'].unique()

# 设置颜色列表
colors = plt.cm.get_cmap('tab10', len(weather_columns))  # 使用tab10颜色表，自动为每个变量分配不同颜色

# 为每个气象变量设定y轴范围
variable_ranges = {
    'asn': (0, 1),
    'sp': (68243, 101110),
    'ssrd': (0, 34000000),
    'str': (-10000000, 80000000),
    'strd': (6000000, 28000000),
    't2m': (238, 286),
    'u10': (-8.5, 8.5),
    'v10': (-6.5, 9.5)
}

# 遍历每个站点
for station in stations:
    # 选择该站点的数据
    station_data = data[data['station'] == station]
    
    # 解析时间列（假设有时间列，如果没有则需要根据实际数据做调整）
    station_data['time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'
    
    # 创建子图，使用len(weather_columns)个子图来分别显示每个变量
    fig, axes = plt.subplots(len(weather_columns), 1, figsize=(10, 6), sharex=True)

    for i, column in enumerate(weather_columns):
        axes[i].plot(station_data['time'], station_data[column], label=column, color=colors(i))
        axes[i].set_title(f"{column} at Station {station}")
        axes[i].set_ylabel("Value")
        axes[i].set_ylim(variable_ranges[column])  # 设置固定的y轴范围
        axes[i].legend()
        axes[i].grid(True)

    # 设置x轴标签，格式化日期为年月
    plt.xlabel("Year-Month")
    fig.autofmt_xdate()  # 自动旋转日期标签

    # 调整布局
    plt.tight_layout()

    # 保存图片到指定路径
    output_path = f"F:\TUB\greenlandProject\ERA5LAND\AllStationFeaturePlot\{station}_weather_data_plot.png"
    plt.savefig(output_path)

    # 关闭当前的图形窗口
    plt.close()

    print(f"图像已保存到: {output_path}")

    # 计算均值
    mean_values = station_data[weather_columns].mean()
    #print(f"\n{station}的均值：")
    #print(mean_values)

    # 检查疑似blunder值（通过计算标准差阈值来进行异常值检测）
    blunder_threshold = 3  # 标准差的倍数
    blunders = {}
    for column in weather_columns:
        column_data = station_data[column]
        mean = column_data.mean()
        std = column_data.std()
        
        # 标记疑似blunder值
        blunder_indices = column_data[(column_data > mean + blunder_threshold * std) | 
                                      (column_data < mean - blunder_threshold * std)].index
        blunders[column] = blunder_indices

    print(f"\n{station}的疑似blunder值索引：")
    for column, indices in blunders.items():
        print(f"{column}: {indices.tolist()}")


C:\Users\11823\AppData\Local\Temp\ipykernel_19936\3812589400.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap('tab10', len(weather_columns))  # 使用tab10颜色表，自动为每个变量分配不同颜色
C:\Users\11823\AppData\Local\Temp\ipykernel_19936\3812589400.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  station_data['time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'


图像已保存到: F:\TUB\greenlandProject\ERA5LAND\AllStationFeaturePlot\CEN_weather_data_plot.png

CEN的疑似blunder值索引：
asn: [77]
sp: []
ssrd: []
str: [99]
strd: [97]
t2m: []
u10: []
v10: []


C:\Users\11823\AppData\Local\Temp\ipykernel_19936\3812589400.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  station_data['time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'


图像已保存到: F:\TUB\greenlandProject\ERA5LAND\AllStationFeaturePlot\CP1_weather_data_plot.png

CP1的疑似blunder值索引：
asn: [173, 234]
sp: [210, 233]
ssrd: []
str: [252]
strd: [253, 254]
t2m: []
u10: []
v10: [110, 186]


C:\Users\11823\AppData\Local\Temp\ipykernel_19936\3812589400.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  station_data['time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'


...

图像已保存到: F:\TUB\greenlandProject\ERA5LAND\AllStationFeaturePlot\ZAC_A_weather_data_plot.png

ZAC_A的疑似blunder值索引：
asn: []
sp: []
ssrd: []
str: []
strd: []
t2m: []
u10: []
v10: []


C:\Users\11823\AppData\Local\Temp\ipykernel_19936\3812589400.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  station_data['time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'


图像已保存到: F:\TUB\greenlandProject\ERA5LAND\AllStationFeaturePlot\ZAC_L_weather_data_plot.png

ZAC_L的疑似blunder值索引：
asn: []
sp: []
ssrd: []
str: []
strd: []
t2m: []
u10: []
v10: []


C:\Users\11823\AppData\Local\Temp\ipykernel_19936\3812589400.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  station_data['time'] = pd.to_datetime(station_data['time'])  # 假设时间列名为 'time'


图像已保存到: F:\TUB\greenlandProject\ERA5LAND\AllStationFeaturePlot\ZAC_U_weather_data_plot.png

ZAC_U的疑似blunder值索引：
asn: []
sp: []
ssrd: []
str: []
strd: []
t2m: []
u10: []
v10: []
